In [4]:
import financedatabase as fd

# Optional Financial Modeling Prep API key for Finance Toolkit functionality
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv("FMP_API_KEY")


In [10]:
all_funds = funds.select()

# Funds have no benchmark field. The objective is only in the name and summary.
description = all_funds["name"].fillna("") + " " + all_funds["summary"].fillna("")

sp500 = r"s\s*&\s*p\s*500|standard\s*(?:&|and)\s*poor'?s?\s*500"
tracks_or_benchmarks = (
    r"track(?:s|ing)?|correspond(?:s|ing)?|replicat(?:e|es|ing)|"
    r"approximat(?:e|es|ing)|benchmark(?:ed)? (?:against|to)|"
    r"total return of the|s\s*&\s*p\s*500 total return"
)
# Other S&P indices, and inverse or leveraged products.
other_index = (
    r"outside the s\s*&\s*p\s*500|excluding the s\s*&\s*p\s*500|"
    r"completion index|extended market|dividend aristocra|"
    r"s\s*&\s*p\s*500\s+(?:buyback|dividend|equal weight|low volatility|"
    r"growth|value|esg|minimum volatil|high dividend)|"
    r"minimum volatil|high dividend|vix|buy-?write|overwrite|"
    r"volatility wtd|volatility-weighted|"
    r"inverse|bear fund|\b-1x\b|\bultra\b|leveraged|\b2x\b|\b3x\b|"
    r"half the return"
)

sp500_total_return = all_funds.loc[
    description.str.contains(sp500, case=False, regex=True)
    & description.str.contains(tracks_or_benchmarks, case=False, regex=True)
    & ~description.str.contains(other_index, case=False, regex=True),
    ["name", "currency", "category_group", "category", "family", "exchange", "summary"],
]

sp500_total_return

,name,currency,category_group,category,family,exchange,summary
symbol,,,,,,,
0P00000JGA.F,ING Direct Fondo Naranja Standard & Poor´S 500 FI,EUR,Equities,Equities,Amundi Asset Management,FRA,ING Direct Fondo Naranja Standard & Poor´S 500...
0P0000X7QE.TO,ivari US Equity Index GIF-GS3 75/75,CAD,Equities,Equities,ivari,TOR,ivari US Equity Index GIF-GS3 75/75 is a segre...
0P0000ZQNS.F,Monument De S&P 500 Tracker EUR,EUR,Equities,Equities,Monument Insurance,FRA,Monument De S&P 500 Tracker EUR is an exchange...
0P0001024D.TO,ivari U.S. Equity Index GIF-NN IP,CAD,Equities,Equities,ivari,TOR,ivari U.S. Equity Index GIF-NN IP is a U.S. eq...
0P0001H0LW.TO,Manulife American Equity Index Fund (U.S.$) Tr...,CAD,Equities,Equities,Manufacturers Life Insurance Co,TOR,Manulife American Equity Index Fund (U.S.$) Tr...
...,...,...,...,...,...,...,...
WFINX,Wells Fargo Index Fund - Class C,USD,Equities,Large Cap,Wells Fargo Funds,NAS,The Wells Fargo Index Fund - Class C seeks to ...
WFIOX,Wells Fargo Index Fund - Class Admin,USD,Equities,Large Cap,Wells Fargo Funds,NAS,The Wells Fargo Index Fund - Class Admin seeks...
WFSPX,iShares S&P 500 Index Fund Class K,USD,Equities,Large Cap,BlackRock Asset Management,NAS,iShares S&P 500 Index Fund Class K aims to tra...


In [ ]:
# No domicile field. US mutual funds here are the USD rows on NAS, with no foreign ticker suffix.
# That drops the Frankfurt, Stuttgart, Madrid, and Toronto listings.
sp500_total_return_us = sp500_total_return.loc[
    (sp500_total_return["currency"] == "USD")
    & (sp500_total_return["exchange"] == "NAS")
    & ~sp500_total_return.index.to_series().str.contains(r"\.", regex=True)
]

sp500_total_return_us